# Prédire le QIT (WPPSI-III) à partir de la HRV néonatale

Le notebook suit les étapes conseillées par le prof :

1. **Preprocessing** : lecture des 3 fichiers, un enregistrement M0 par enfant, nettoyage, enfants et plis communs à l'équipe.
2. **Feature selection** : moins de variables que d'enfants, par importance d'une forêt aléatoire ou par PCA (90 % de variance), toujours à l'intérieur des plis.
3. **Models** : Elastic Net, Random Forest, LightGBM, XGBoost, SVR.
4. **Evaluation** : T × K validation croisée (T = 10 répétitions de K = 5 plis) avec `evaluate_model.py`, le script commun de l'équipe.
5. **Prédiction conforme** : un intervalle qui contient le vrai QIT avec une probabilité d'au moins 1 − α.
6. **Explication** : quelles variables comptent pour le modèle, et ce qu'on peut en conclure ou non.

**Avant de lancer**
1. Mettre les 3 CSV dans `data/` et installer les dépendances : `pip install -r requirements.txt`.
2. Lancer une première fois avec `RAPIDE = True` (quelques minutes) et lire les messages de vérification de chaque section.
3. Passer `RAPIDE = False` pour les chiffres à rapporter (environ 15 minutes).

**Repères dans le code** (Ctrl+F)
- `[À ADAPTER]` : à vérifier sur vos fichiers. Ce code a été écrit sans voir les données, à partir de leur description.
- `[PISTE]` : réglage qui peut améliorer le modèle, à tester une fois que tout tourne.

**Données de santé d'enfants** : `cohort.csv`, `folds.csv` et les prédictions sont écrits dans `data/cleaned/`, ignoré par git. Vider les sorties du notebook avant tout commit (*Restart & Clear Output*), car le dépôt est public : `pip install nbstripout` puis `nbstripout --install` le fait automatiquement. Les sorties n'affichent que des effectifs, des noms de colonnes et des métriques globales, sauf les graphiques individuels signalés en 4.4 et 5.

In [ ]:
from pathlib import Path
import re
import sys
import time
import unicodedata
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from lightgbm import LGBMRegressor
from scipy.stats import kruskal, rankdata
from xgboost import XGBRegressor

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import ElasticNetCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PowerTransformer, QuantileTransformer, StandardScaler
from sklearn.svm import SVR

# ------------------------------------------------------------------ Fichiers
RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RACINE))
from evaluate_model import evaluate, load_labels, norm_id, run_cv   # script commun de l'équipe (racine du dépôt)

DOSSIER_DONNEES = RACINE / "data"
DOSSIER_PARTAGE = DOSSIER_DONNEES / "cleaned"   # cohort.csv, folds.csv et prédictions (ignorés par git)
FICHIER_HRV_LIN = "HRV_Lineaire_Full.csv"
FICHIER_HRV_NL = "HRV_NonLineaire_Full.csv"
FICHIER_WPPSI = "Résultats WPPSI III.csv"

# ------------------------------------------------------------------ Colonnes
# La recherche des colonnes ignore majuscules, accents et espaces. Un accent devenu illisible dans le
# fichier (« Num�ro ») est accepté à la place de n'importe quelle lettre.
COL_ID_HRV = "Numéro"
COL_ID_WPPSI = "N°"
COL_PASSAGE = "Passage"
PASSAGE_RETENU = "M0"              # [PISTE] M6, M12... : la HRV plus tardive prédit-elle mieux ?
# Le CSV WPPSI a deux lignes d'en-tête : les blocs (NS Verb., NS Perf., Note composite, Rang percentile,
# Intervalle de confiance) au-dessus des colonnes (N°, DDN, sous-tests, QIV / QIP / QIT / NCL).
# Les deux sont réunies en « Bloc | Colonne », par exemple « Note composite | QIT ».
LIGNES_ENTETE_WPPSI = 2
LIGNES_ENTETE_HRV = 1              # [À ADAPTER] 2 si les fichiers HRV ont aussi deux lignes d'en-tête
COLS_SCORES = {"QIV": "Note composite | QIV", "QIP": "Note composite | QIP", "QIT": "Note composite | QIT"}
CIBLES = ["QIT"]                   # [PISTE] ["QIV", "QIP", "QIT"] : la HRV est peut-être liée à un domaine précis
# Covariables cliniques du fichier HRV linéaire (enfant et grossesse).
# [PISTE] « Sortie du service » (unité de sortie ?) pourrait s'ajouter si c'est une catégorie et non une date.
COLS_CLINIQUES = ["Sexe", "Prématuré", "SFA", "Réa à la naissance", "Pathologie neurologique",
                  "Tabac", "Diabète", "HTA", "Maturation Pulm", "Dépression: prise d'IRS"]
# Qualité de l'enregistrement = part des RR analysés (noms du fichier linéaire, puis du non linéaire).
COLS_RR_ANALYSES = ["nb RR analyzed total", "nb analyzed RR total"]
COL_RR_TOTAL = "nb RR total"
QUALITE_MIN = None                 # [PISTE] ex. 70 : écarte les enregistrements où moins de 70 % des RR sont analysés

# Colonnes jamais utilisées comme variables d'entrée : identité, dates, horaires, durées, nombres de RR, scores WPPSI.
# [À ADAPTER] Relire la liste « exclues » affichée en 1.4 et compléter si une colonne passe à travers.
MOTIFS_EXCLUS = [r"numero", r"^n°", r"monogramme", r"ddn", r"filename", r"fichier", r"recorder", r"date",
                 r"heure", r"duree", r"duration", r"beginning", r"\b(debut|fin|start|end)\b", r"nb ?rr",
                 r"analy[sz]ed", r"passage", r"sortie", r"^(qit|qiv|qip|ncl|com|inf|dim|cub|aob)\b"]

# ------------------------------------------------------------------ Preprocessing
MAX_MANQUANTS_COL = 0.30           # [PISTE] colonnes avec plus de 30 % de valeurs manquantes retirées
MAX_MANQUANTS_LIGNE = 0.50         # [PISTE] enfants avec plus de 50 % d'indices manquants retirés
MIN_CAS_RARES = 5                  # [PISTE] covariable binaire retirée si une des deux classes a moins de 5 enfants
PERIODES = ("24h", "Day", "Night") # [PISTE] ("Night",) : sommeil seul ; ("24h",) : trois fois moins de variables
AJOUTER_NUIT_MOINS_JOUR = False    # [PISTE] ajoute Nuit − Jour pour chaque indice (réactivité circadienne)

# ------------------------------------------------------------------ Feature selection
SEUIL_CORR = 0.95                  # [PISTE] 0.90 puis 0.80 : moins de variables redondantes = modèle plus stable
TRANSFO = "quantile"               # [PISTE] "yeo-johnson" (environ 20 fois plus lent) pour vérifier que le résultat n'en dépend pas
TOP_K = 30                         # [PISTE] nombre de variables gardées par l'importance de la forêt (15, 30, 50)
VARIANCE_PCA = 0.90                # [PISTE] part de variance gardée par la PCA (0.80, 0.90, 0.95)

# ------------------------------------------------------------------ Evaluation
RAPIDE = True                      # [À ADAPTER] False pour les résultats à rapporter
T_REPETITIONS, K_PLIS = 10, 5      # plis communs : T répétitions de K plis, soit T × K = 50 mesures par modèle
REGENERER_PLIS = False             # [À ADAPTER] True pour recréer cohort.csv / folds.csv (après un changement de filtre)
REPETITIONS = [1, 2] if RAPIDE else None   # None = les T répétitions de folds.csv
N_PERMUTATIONS = 20 if RAPIDE else 100
ALPHA = 0.10                       # prédiction conforme : intervalle à 1 − α = 90 %
GRAINE = 42
N_JOBS = -1                        # tous les cœurs du processeur

# ------------------------------------------------------------------ Elastic Net
# Vocabulaire : dans sklearn, `alpha` est la force de la pénalité (λ dans glmnet / R) et `l1_ratio`
# le mélange L1/L2 (α dans glmnet). l1_ratio = 1 donne un Lasso, l1_ratio proche de 0 un Ridge.
L1_RATIOS = [0.05, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0]   # [PISTE] resserrer autour des valeurs choisies (section 6)
N_ALPHAS = 30
EPS_ALPHA = 1e-2                   # plus petit alpha testé = alpha_max × EPS_ALPHA ([PISTE] 1e-3 si alpha choisi est souvent le plus petit)

# 1. Preprocessing

## 1.1 Lecture des fichiers

In [ ]:
def normaliser_texte(s):
    """Minuscules, sans accents, espaces simplifiés : ' Numéro ' -> 'numero'."""
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", s).strip().lower()


def cle(s):
    return re.sub(r"[\s_.\-]", "", normaliser_texte(s))


def meme_nom(colonne, nom):
    """Compare les clés ; un caractère illisible « � » dans le nom de colonne vaut n'importe quel caractère."""
    motif = "".join("." if ch == "\ufffd" else re.escape(ch) for ch in cle(colonne))
    return re.fullmatch(motif, cle(nom)) is not None


def trouver_colonne(df, *noms):
    """Vrai nom de la colonne de df qui correspond à l'un des noms (ou à la fin d'un nom « Bloc | Nom »), sinon None."""
    for nom in noms:
        exactes = [c for c in df.columns if meme_nom(c, nom)]
        if exactes:
            return exactes[0]
        fins = [c for c in df.columns if meme_nom(str(c).split("|")[-1], nom)]
        if len(fins) == 1:
            return fins[0]
    return None


def aplatir_entetes(colonnes):
    """Deux lignes d'en-tête (cellules fusionnées d'Excel) -> un nom « Bloc | Sous-colonne »."""
    noms, bloc = [], ""
    for haut, bas in colonnes:
        haut = "" if str(haut).startswith("Unnamed") else str(haut).strip()
        bas = "" if str(bas).startswith("Unnamed") else str(bas).strip()
        bloc = haut or bloc        # une cellule fusionnée n'est remplie que sur sa première colonne
        noms.append(f"{bloc} | {bas}" if bloc and bas else bloc or bas)
    return noms


LETTRES_FR = set("éèêëàâäùûüçîïôö°")
UNITES = {"-", "n", "ms", "ms^2", "ms2", "ms/beat", "bpm", "%", "hz", "s", "hh:mm:ss", "u.a.", "n.u.", "0/1", "m/f", "unit"}
VALEURS_VIDES = {"", "-", "--", "na", "n/a", "nan", "nd", "?", "#n/a", "#div/0!", "#valeur!", "#value!", "#ref!", "#nombre!"}


def detecter_encodage(brut):
    """UTF-16, UTF-8, Windows (cp1252) ou Excel Mac (mac_roman)."""
    if brut[:2] in (b"\xff\xfe", b"\xfe\xff"):
        return "utf-16"
    try:
        brut.decode("utf-8-sig")
        return "utf-8-sig"
    except UnicodeDecodeError:
        pass
    # Windows et Mac codent « é » différemment : on garde celui qui produit le plus de lettres françaises.
    return max(["cp1252", "mac_roman"], key=lambda e: sum(ch in LETTRES_FR for ch in brut.decode(e, errors="replace")))


def reparer_accents(nom):
    """'NumÃ©ro' (UTF-8 relu comme Windows) -> 'Numéro'."""
    try:
        return nom.encode("cp1252").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return nom


def lire_csv(chemin, lignes_entete=1):
    """Lit un CSV exporté d'Excel : encodage, séparateur, virgule décimale, ligne « sep=; » et ligne d'unités détectés."""
    brut = Path(chemin).read_bytes()
    encodage = detecter_encodage(brut)
    lignes = [l for l in brut.decode(encodage, errors="replace").splitlines() if l.strip()]
    a_sauter = 1 if lignes[0].strip().lower().startswith("sep=") else 0   # ligne ajoutée par certains exports Excel
    sep = max([";", ",", "\t"], key=lignes[a_sauter].count)
    df = pd.read_csv(chemin, sep=sep, encoding=encodage, encoding_errors="replace", skiprows=a_sauter,
                     header=list(range(lignes_entete)), decimal="," if sep != "," else ".")
    if lignes_entete > 1:
        df.columns = aplatir_entetes(df.columns)
    noms, vus = [], {}
    for c in df.columns:
        nom = reparer_accents(str(c).strip()) or "sans_nom"      # colonne sans en-tête
        vus[nom] = vus.get(nom, 0) + 1
        noms.append(nom if vus[nom] == 1 else f"{nom}.{vus[nom] - 1}")
    df.columns = noms
    lecture = f"encodage {encodage}, séparateur {sep!r}" + (", ligne « sep= » ignorée" if a_sauter else "")
    ligne_1 = df.iloc[0].astype(str).tolist() if len(df) else []   # avant conversion, pour le diagnostic d'en-tête
    # Ligne d'unités / légendes sous l'en-tête (M/F, 0/1, ms, bpm, %...) : ce ne sont pas des données.
    valeurs = [normaliser_texte(v) for v in (df.iloc[0] if len(df) else []) if pd.notna(v) and str(v).strip()]
    if valeurs and sum(v in UNITES for v in valeurs) >= 0.5 * len(valeurs):
        df = df.iloc[1:].reset_index(drop=True)
        lecture += ", ligne d'unités ignorée"
    df.attrs["lecture"], df.attrs["ligne_1"] = lecture, ligne_1
    # Colonnes numériques restées en texte (virgule décimale, %, espaces, « - » ou « NA » pour une case vide) :
    # converties si au moins 90 % des valeurs renseignées sont des nombres.
    presque = {}
    for c in df.columns:
        if not pd.api.types.is_numeric_dtype(df[c]) and df[c].notna().any():
            texte = df[c].astype("string").str.replace(r"[\s\u00a0\u202f%]", "", regex=True).str.replace(",", ".")
            texte = texte.where(~texte.str.lower().isin(VALEURS_VIDES).fillna(False).astype(bool))
            nombres = pd.to_numeric(texte, errors="coerce").astype("float64")
            n_remplies, n_nombres = texte.notna().sum(), nombres.notna().sum()
            if n_nombres >= 0.9 * n_remplies:
                df[c] = nombres
            elif n_nombres >= 0.5 * n_remplies:   # presque numérique : on note ce qui bloque pour le diagnostic
                presque[c] = (f"{n_nombres / n_remplies:.0%} de nombres",
                              texte[nombres.isna() & texte.notna()].str.slice(0, 12).value_counts().head(3).index.tolist())
    df.attrs["presque_numeriques"] = presque
    return df


hrv_lin_brut = lire_csv(DOSSIER_DONNEES / FICHIER_HRV_LIN, LIGNES_ENTETE_HRV)
hrv_nl_brut = lire_csv(DOSSIER_DONNEES / FICHIER_HRV_NL, LIGNES_ENTETE_HRV)
wppsi_brut = lire_csv(DOSSIER_DONNEES / FICHIER_WPPSI, LIGNES_ENTETE_WPPSI)

for nom, df in [("HRV linéaire", hrv_lin_brut), ("HRV non linéaire", hrv_nl_brut), ("WPPSI", wppsi_brut)]:
    n_num = sum(pd.api.types.is_numeric_dtype(df[c]) for c in df.columns)
    print(f"{nom:17s}: {df.shape[0]} lignes, {df.shape[1]} colonnes dont {n_num} numériques ({df.attrs['lecture']})")
    if df.attrs["presque_numeriques"]:
        # [À ADAPTER] Colonnes restées en texte à cause de quelques valeurs : ajouter ces valeurs à VALEURS_VIDES
        # si ce sont des marqueurs de case vide, ou corriger le fichier.
        print(f"   presque numériques mais restées en texte (valeurs qui bloquent) : "
              f"{dict(list(df.attrs['presque_numeriques'].items())[:10])}")

# [À ADAPTER] Lire ces listes une fois : elles disent si les noms de la configuration sont les bons.
print("\nColonnes WPPSI :", list(wppsi_brut.columns))
print("\nColonnes texte des fichiers HRV :",
      [c for df in (hrv_lin_brut, hrv_nl_brut) for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])])

## 1.2 Identifiants

L'identifiant commun est le **numéro** du sujet (1re colonne de chaque fichier), harmonisé comme dans `evaluate_model.py` : on garde le premier nombre (« 012 », « 12.0 », « 12-ABC-M0 » donnent « 12 »). Les initiales ne suffisent pas, car plusieurs enfants ont les mêmes (voir `data/pulisci_righe.py`). La date de naissance sert de vérification en 1.5. Seuls les formats sont affichés (`A` = lettre, `9` = chiffre), jamais les identifiants eux-mêmes.

In [ ]:
def format_id(s):
    return re.sub(r"\d", "9", re.sub(r"[A-Za-z]", "A", s))


# Noms d'identifiant reconnus si le nom configuré est absent : N°, Nº, No, Num, Numéro, ID, Code
# (éventuellement suivis de patient / sujet / enfant / aube / dossier), Patient, Sujet, Enfant.
MOTIF_ID = re.compile(r"^((n°|no|num|numero|id|identifiant|code)(patient|sujet|enfant|aube|dossier)?|patient|sujet|enfant)$")


def trouver_id(df, nom_configure):
    col = trouver_colonne(df, nom_configure)
    if col is None:
        candidates = [c for c in df.columns if MOTIF_ID.match(cle(c))]
        col = candidates[0] if candidates else None
    return col


fichiers = {"lin": hrv_lin_brut, "nl": hrv_nl_brut, "wppsi": wppsi_brut}
col_id = {nom: trouver_id(df, COL_ID_WPPSI if nom == "wppsi" else COL_ID_HRV) for nom, df in fichiers.items()}
print("Colonne identifiant :", col_id)
if None in col_id.values():
    for nom, df in fichiers.items():
        if col_id[nom] is None:
            print(f"\n{nom} ({df.attrs['lecture']}), {df.shape[1]} colonnes, les premières : {list(df.columns[:20])}")
            if df.shape[1] <= 2:
                print("  -> presque aucune colonne : le séparateur ou la ligne d'en-tête n'ont pas été reconnus")
            if any(MOTIF_ID.match(cle(v)) for v in df.attrs["ligne_1"]):
                print("  -> le nom de l'identifiant est sur la 1re ligne de données : l'en-tête tient sur 2 lignes "
                      "(mettre LIGNES_ENTETE_HRV ou LIGNES_ENTETE_WPPSI à 2)")
    raise KeyError("Colonne identifiant introuvable : mettre son nom exact (voir la liste ci-dessus) "
                   "dans COL_ID_HRV / COL_ID_WPPSI.")

for nom, df in fichiers.items():
    df["id_enfant"] = df[col_id[nom]].map(norm_id)          # même règle que evaluate_model.py
    formats = df[col_id[nom]].dropna().astype(str).str.strip().map(format_id).value_counts().head(3).to_dict()
    print(f"  {nom:6s} {df['id_enfant'].nunique()} identifiants, formats les plus fréquents : {formats}")
communs = set(hrv_lin_brut["id_enfant"].dropna()) & set(hrv_nl_brut["id_enfant"].dropna()) & set(wppsi_brut["id_enfant"].dropna())
print(f"Identifiants présents dans les 3 fichiers : {len(communs)}")

## 1.3 Scores WPPSI

Les colonnes QIV, QIP, NCL et les sous-tests (COM, INF, DIM, CUB, AOB) viennent du même test que le QIT : elles ne sont jamais utilisées comme variables d'entrée. QIV, QIP et QIT sont gardés comme cibles possibles, comme dans `evaluate_model.py`.

In [ ]:
wppsi = wppsi_brut.dropna(subset=["id_enfant"])

# [À ADAPTER] Si le fichier a une ligne par échelle (une colonne vaut QIV / QIP / QIT...) au lieu d'une
# ligne par enfant, le remettre d'abord en une ligne par enfant, par exemple :
# wppsi = wppsi.pivot_table(index="id_enfant", columns="Échelle", values="Note composite", aggfunc="first").reset_index()


def resume_numerique(df, colonnes):
    lignes = {c: df[c].describe()[["count", "min", "50%", "max"]] for c in colonnes if pd.api.types.is_numeric_dtype(df[c])}
    return pd.DataFrame(lignes).T.rename(columns={"count": "n", "50%": "médiane"})


def trouver_score(echelle, nom_configure):
    """Colonne de la note composite d'une échelle (QIV, QIP, QIT), sinon None."""
    col = trouver_colonne(wppsi, nom_configure)
    if col is not None and pd.api.types.is_numeric_dtype(wppsi[col]):
        return col
    # Colonne numérique dont le nom contient l'échelle et dont la médiane ressemble à un QI
    # (le rang percentile tourne autour de 50).
    candidates = [c for c in wppsi.columns if echelle.lower() in normaliser_texte(c) and pd.api.types.is_numeric_dtype(wppsi[c])]
    plausibles = [c for c in candidates if 70 <= wppsi[c].median() <= 130]
    return plausibles[0] if len(plausibles) == 1 else None


cols_scores = {e: trouver_score(e, nom) for e, nom in COLS_SCORES.items()}
print("Colonnes des scores :", cols_scores)
if cols_scores["QIT"] is None:
    raise ValueError("QIT introuvable : mettre le bon nom dans COLS_SCORES (ou LIGNES_ENTETE_WPPSI = 2 si « QIT » "
                     "est sur la 2e ligne d'en-tête du CSV).")
cols_scores = {e: c for e, c in cols_scores.items() if c is not None}
print(resume_numerique(wppsi, list(cols_scores.values())))   # [À ADAPTER] vérifier : entre ~40 et ~160, médiane proche de 100

# Plusieurs lignes pour un enfant : on garde la première valeur non vide. Si elles donnent deux QIT
# différents, l'enfant est retiré, faute de savoir lequel est le bon.
scores = wppsi[["id_enfant", *cols_scores.values()]].rename(columns={c: e for e, c in cols_scores.items()})
n_valeurs = scores.dropna(subset=["QIT"]).groupby("id_enfant")["QIT"].nunique()
conflits = n_valeurs.index[n_valeurs > 1]
print(f"{wppsi['id_enfant'].duplicated().sum()} lignes en double, {len(conflits)} enfants aux QIT contradictoires retirés")
scores = scores[~scores["id_enfant"].isin(conflits)].groupby("id_enfant").first()
# Enfants sans tous les scores retirés, comme dans data/pulisci_righe.py.
incomplets = scores.isna().any(axis=1)
hors_bornes = ~incomplets & ~scores.apply(lambda s: s.between(40, 160)).all(axis=1)
print(f"{incomplets.sum()} enfants sans tous les scores ({', '.join(scores.columns)}), "
      f"{hors_bornes.sum()} avec un score hors de [40, 160] : retirés")
scores = scores[~incomplets & ~hors_bornes]
print(f"{len(scores)} enfants avec leurs scores : QIT {scores['QIT'].mean():.1f} ± {scores['QIT'].std():.1f}, "
      f"{(scores['QIT'] < 90).mean():.0%} sous 90")

## 1.4 Indices HRV : un enregistrement M0 par enfant

In [ ]:
def normaliser_passage(valeur):
    """'MO' (lettre O), 'm0', 'M00' -> 'M0' ; 'M06' -> 'M6'."""
    s = re.sub(r"\s", "", str(valeur)).upper()
    s = "M0" if s == "MO" else s
    return re.sub(r"^M0+(?=\d)", "M", s)


def periode(colonne):
    """'SDNN (TDNight)' -> 'Night', 'Higuchi (24h)' -> '24h', sans période -> None."""
    m = re.search(r"(24h|Day|Night)\)$", colonne)
    return m.group(1) if m else None


def preparer_hrv(brut, bloc):
    """Garde le passage retenu et une ligne par enfant, puis sépare les indices HRV du reste."""
    df = brut.dropna(subset=["id_enfant"])
    col_passage = trouver_colonne(df, COL_PASSAGE)
    if col_passage is None:
        print(f"[{bloc}] pas de colonne « {COL_PASSAGE} » : on suppose déjà une ligne par enfant")
    else:
        passage = df[col_passage].map(normaliser_passage)
        print(f"[{bloc}] passages : {passage.value_counts().to_dict()}")
        df = df[passage == normaliser_passage(PASSAGE_RETENU)]   # [À ADAPTER] si M0 est codé autrement (« naissance »...)

    col_ana, col_tot = trouver_colonne(df, *COLS_RR_ANALYSES), trouver_colonne(df, COL_RR_TOTAL)
    if col_ana is not None and col_tot is not None and not all(pd.api.types.is_numeric_dtype(df[c]) for c in (col_ana, col_tot)):
        print(f"[{bloc}] « {col_ana} » ou « {col_tot} » n'est pas numérique : qualité non calculée "
              "(voir les colonnes « presque numériques » en 1.1)")
    elif col_ana is not None and col_tot is not None:
        qualite = 100 * df[col_ana] / df[col_tot]
        print(f"[{bloc}] RR analysés : médiane {qualite.median():.0f} %, minimum {qualite.min():.0f} %")
        if QUALITE_MIN is not None:
            df, qualite = df[qualite >= QUALITE_MIN], qualite[qualite >= QUALITE_MIN]
        df = df.loc[qualite.sort_values(ascending=False).index]   # en cas de doublon, on garde le mieux analysé
    n_doublons = df["id_enfant"].duplicated().sum()
    df = df.drop_duplicates("id_enfant").set_index("id_enfant")
    print(f"[{bloc}] {len(df)} enfants au passage {PASSAGE_RETENU}, {n_doublons} enregistrements en double écartés")

    cliniques = [c for c in (trouver_colonne(df, n) for n in COLS_CLINIQUES) if c is not None]
    exclues = [c for c in df.columns
               if c in cliniques or c == col_id[bloc] or any(re.search(m, normaliser_texte(c)) for m in MOTIFS_EXCLUS)]
    texte = [c for c in df.columns if c not in exclues and not pd.api.types.is_numeric_dtype(df[c])]
    indices = [c for c in df.columns if c not in exclues and c not in texte]
    print(f"[{bloc}] {len(indices)} indices HRV ; exclues : {exclues}")
    if texte:
        print(f"[{bloc}] colonnes texte ignorées : {texte}")
    meta = df.drop(columns=indices)

    indices = [c for c in indices if periode(c) in (None, *PERIODES)]
    hrv = df[indices]
    if AJOUTER_NUIT_MOINS_JOUR:
        nuit_jour = {c[:-4] + "Nuit-Jour)": df[c[:-4] + "Night)"] - df[c]
                     for c in indices if c.endswith("Day)") and c[:-4] + "Night)" in indices}
        hrv = pd.concat([hrv, pd.DataFrame(nuit_jour)], axis=1)
    print(f"[{bloc}] {hrv.shape[1]} variables gardées (périodes {PERIODES}"
          + (", avec Nuit − Jour)" if AJOUTER_NUIT_MOINS_JOUR else ")"))
    return hrv.add_prefix(f"{bloc}|"), meta


hrv_lin, meta_lin = preparer_hrv(hrv_lin_brut, "lin")
hrv_nl, meta_nl = preparer_hrv(hrv_nl_brut, "nl")

## 1.5 Fusion, vérification des dates de naissance, covariables cliniques, nettoyage

In [ ]:
# Seuls les enfants présents dans les 3 fichiers sont gardés : un enfant qui a un QIT mais pas d'enregistrement
# M0 dans les deux fichiers HRV n'a pas les mêmes informations que les autres.
hrv = hrv_lin.join(hrv_nl, how="inner")
donnees = hrv.join(scores, how="inner")
print(f"HRV linéaire {len(hrv_lin)}, HRV non linéaire {len(hrv_nl)}, les deux {len(hrv)}, avec les scores {len(donnees)} enfants")
print(f"  {len(scores) - len(donnees)} enfants avec des scores mais sans enregistrement {PASSAGE_RETENU} dans les deux fichiers HRV : retirés")

# Même numéro = même enfant ? On compare les dates de naissance des 3 fichiers.
wppsi_enfant = wppsi.groupby("id_enfant").first()


def dates_naissance(source):
    col = trouver_colonne(source, "DDN")
    if col is None:
        return None
    texte = source[col].astype("string").str.strip()
    return pd.to_datetime(texte, dayfirst=True, errors="coerce", format="mixed").reindex(donnees.index)


ddn = pd.DataFrame({nom: d for nom, source in [("lin", meta_lin), ("nl", meta_nl), ("wppsi", wppsi_enfant)]
                    if (d := dates_naissance(source)) is not None})
differentes = (ddn.notna().sum(axis=1) >= 2) & (ddn.nunique(axis=1) > 1)
print(f"{differentes.sum()} enfants dont la date de naissance diffère d'un fichier à l'autre : retirés "
      f"(dates lisibles pour {ddn.notna().all(axis=1).mean():.0%} des enfants)")   # [À ADAPTER] les vérifier avec l'équipe
donnees = donnees[~differentes]

# Covariables cliniques, prises dans le premier fichier qui les contient.
clinique = pd.DataFrame(index=donnees.index)
for nom in COLS_CLINIQUES:
    for source in [meta_lin, meta_nl, wppsi_enfant]:
        c = trouver_colonne(source, nom)
        if c is not None:
            valeurs = source[c].reindex(donnees.index)
            if not pd.api.types.is_numeric_dtype(valeurs):
                valeurs = valeurs.astype(object).where(valeurs.notna(), np.nan)
            clinique[f"clin|{nom}"] = valeurs
            break
introuvables = [n for n in COLS_CLINIQUES if f"clin|{n}" not in clinique.columns]
if introuvables:
    print(f"Covariables cliniques introuvables : {introuvables}")   # [À ADAPTER] sont-elles dans un autre fichier ?
# Une covariable binaire avec très peu de cas (ex. 3 enfants) ne peut pas être apprise de façon fiable.
rares = [c for c in clinique.columns
         if clinique[c].nunique() < 2 or (clinique[c].nunique() == 2 and clinique[c].value_counts().min() < MIN_CAS_RARES)]
if rares:
    print(f"Covariables constantes ou trop rares retirées : {rares}")
donnees = donnees.join(clinique.drop(columns=rares))

# Nettoyage sans regarder le QIT, donc sans fuite vers la cible.
cols_hrv = list(hrv.columns)
donnees[cols_hrv] = donnees[cols_hrv].replace([np.inf, -np.inf], np.nan)
taux_manquants = donnees[cols_hrv].isna().mean()
a_retirer = taux_manquants.index[(taux_manquants > MAX_MANQUANTS_COL) | (donnees[cols_hrv].nunique() <= 1)]
donnees = donnees.drop(columns=a_retirer)
incomplets = donnees[[c for c in cols_hrv if c not in a_retirer]].isna().mean(axis=1) > MAX_MANQUANTS_LIGNE
donnees = donnees[~incomplets]
print(f"{len(a_retirer)} colonnes trop vides ou constantes retirées, {incomplets.sum()} enfants trop incomplets retirés")

## 1.6 Enfants et plis communs à l'équipe

`evaluate_model.py` ne compare deux modèles que s'ils ont prédit **les mêmes enfants avec les mêmes plis**. Ces deux fichiers doivent venir de `prepare_data.py` de l'équipe : `cohort.csv` (child_id, QIV, QIP, QIT) et `folds.csv` (child_id, repeat, fold). Tant qu'il n'est pas disponible, le notebook les crée au même format, avec T = 10 répétitions de K = 5 plis stratifiés sur le QIT. Quand l'équipe fournit les siens, les mettre dans `data/cleaned/` : ils seront utilisés tels quels.

In [ ]:
fichier_cohorte, fichier_plis = DOSSIER_PARTAGE / "cohort.csv", DOSSIER_PARTAGE / "folds.csv"
if REGENERER_PLIS or not (fichier_cohorte.exists() and fichier_plis.exists()):
    DOSSIER_PARTAGE.mkdir(parents=True, exist_ok=True)
    donnees[list(scores.columns)].rename_axis("child_id").reset_index().to_csv(fichier_cohorte, index=False)
    tranches = pd.qcut(donnees["QIT"], q=5, labels=False, duplicates="drop")   # chaque pli a des QIT bas, moyens et hauts
    plis = [pd.DataFrame({"child_id": donnees.index[te], "repeat": r, "fold": k})
            for r in range(1, T_REPETITIONS + 1)
            for k, (_, te) in enumerate(StratifiedKFold(K_PLIS, shuffle=True, random_state=GRAINE + r)
                                        .split(donnees, tranches), start=1)]
    pd.concat(plis).to_csv(fichier_plis, index=False)
    print(f"cohort.csv et folds.csv créés : {len(donnees)} enfants, {T_REPETITIONS} × {K_PLIS} plis ({DOSSIER_PARTAGE})")
else:
    print(f"cohort.csv et folds.csv existants utilisés ({DOSSIER_PARTAGE})")

plis_partages = pd.read_csv(fichier_plis, dtype={"child_id": str})
plis_partages["child_id"] = plis_partages["child_id"].map(norm_id)
etiquettes = load_labels(fichier_cohorte, CIBLES)            # scores de la cohorte commune, une colonne par cible
sans_variables = set(plis_partages["child_id"]) - set(donnees.index)
if sans_variables:
    raise ValueError(f"{len(sans_variables)} enfants de folds.csv n'ont pas de variables ici. Si ces fichiers viennent "
                     "d'un ancien passage du notebook, mettre REGENERER_PLIS = True ; s'ils viennent de l'équipe, "
                     "comparer les deux nettoyages.")
hors_cohorte = len(donnees) - plis_partages["child_id"].nunique()
if hors_cohorte:
    print(f"{hors_cohorte} enfants préparés ici ne sont pas dans la cohorte commune : non utilisés")
donnees = donnees.loc[sorted(plis_partages["child_id"].unique(), key=int)]

X = donnees.drop(columns=list(scores.columns))
y = donnees["QIT"].to_numpy(dtype=float)
cols_lin = [c for c in X.columns if c.startswith("lin|")]
cols_nl = [c for c in X.columns if c.startswith("nl|")]
cols_clin = [c for c in X.columns if c.startswith("clin|")]
CLIN_TEXTE = {c for c in cols_clin if not pd.api.types.is_numeric_dtype(X[c])}
print(f"Jeu final : {len(y)} enfants, {len(cols_lin)} indices linéaires, {len(cols_nl)} non linéaires, "
      f"{len(cols_clin)} cliniques ({X.shape[1] / len(y):.1f} variables par enfant)")

## 1.7 Contrôles

In [ ]:
# Effet d'appareil : si le QIT moyen diffère selon l'enregistreur, un modèle peut apprendre l'appareil plutôt
# que la physiologie (par exemple si un appareil a surtout servi en néonatologie).
col_appareil = trouver_colonne(meta_lin, "Recorder")
if col_appareil is not None:
    appareil = meta_lin[col_appareil].reindex(donnees.index)
    print(donnees.groupby(appareil)["QIT"].agg(["count", "mean"]).round(1))
    groupes = [g["QIT"].to_numpy() for _, g in donnees.groupby(appareil) if len(g) >= 5]
    if len(groupes) >= 2:
        # [PISTE] si p < 0.05, regarder aussi les indices HRV par appareil et ajouter l'appareil en covariable
        print(f"Kruskal-Wallis QIT selon l'appareil : p = {kruskal(*groupes).pvalue:.3f}")

# Repère : plus forte corrélation d'une variable avec le QIT, comparée à ce que donne le hasard avec autant de
# variables. Descriptif seulement : choisir des variables ici, hors des plis, serait une fuite d'information.
numeriques = X.select_dtypes("number")
rho = numeriques.corrwith(donnees["QIT"], method="spearman").abs().sort_values(ascending=False)
melange = np.random.default_rng(0)
rho_hasard = [numeriques.corrwith(pd.Series(melange.permutation(y), index=X.index), method="spearman").abs().max()
              for _ in range(20)]
print(f"\n|Spearman| le plus fort avec le QIT : {rho.iloc[0]:.2f} ; par pur hasard avec {numeriques.shape[1]} variables, "
      f"on obtient en moyenne {np.mean(rho_hasard):.2f}")
print(rho.head(10).round(2).to_string())

# 2. Feature selection

Avec environ 200 variables pour 180 enfants, un modèle peut apprendre le bruit des enfants d'entraînement au lieu d'un vrai lien (surapprentissage). On réduit donc le nombre de variables, **à l'intérieur de chaque pli, sur les seuls enfants d'entraînement** : les choisir sur tous les enfants avant la validation croisée donnerait des performances trop optimistes.

Dans chaque pli, dans l'ordre :
1. **Prétraitement** (toujours) : imputation par la médiane, retrait des variables corrélées à plus de `SEUIL_CORR`, transformation qui réduit l'asymétrie et met toutes les variables à la même échelle.
2. **Sélection**, au choix pour chaque expérience :
   - `"toutes"` : pas de sélection, le modèle fait lui-même le tri (Elastic Net met des coefficients à zéro, les forêts choisissent leurs coupures) ;
   - `"sél. RF"` : une forêt aléatoire de référence est entraînée et on garde ses `TOP_K` variables les plus importantes ;
   - `"PCA 90 %"` : les variables sont remplacées par le plus petit nombre de composantes principales qui gardent 90 % de leur variance.

In [ ]:
class FiltreCorrelation(BaseEstimator, TransformerMixin):
    """Garde une variable par groupe de variables très corrélées (|Spearman| > seuil), ajusté dans chaque pli."""

    def __init__(self, seuil=0.95):
        self.seuil = seuil

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.n_features_in_ = X.shape[1]
        if self.seuil is None or X.shape[1] < 2:
            self.garder_ = np.arange(X.shape[1])
            return self
        with np.errstate(invalid="ignore", divide="ignore"):
            corr = np.abs(np.nan_to_num(np.corrcoef(rankdata(X, axis=0), rowvar=False)))
        garder = []
        for j in range(X.shape[1]):
            if not garder or corr[j, garder].max() <= self.seuil:
                garder.append(j)
        self.garder_ = np.array(garder)
        return self

    def transform(self, X):
        return np.asarray(X, dtype=float)[:, self.garder_]

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = [f"x{i}" for i in range(self.n_features_in_)]
        return np.asarray(input_features, dtype=object)[self.garder_]


def faire_transfo():
    if TRANSFO == "yeo-johnson":
        return PowerTransformer(method="yeo-johnson")   # réduit l'asymétrie (puissances spectrales...) puis standardise
    if TRANSFO == "quantile":
        return QuantileTransformer(n_quantiles=50, output_distribution="normal")
    return StandardScaler()


def faire_preprocesseur(colonnes):
    hrv = [c for c in colonnes if not c.startswith("clin|")]
    clin_num = [c for c in colonnes if c.startswith("clin|") and c not in CLIN_TEXTE]
    clin_texte = [c for c in colonnes if c in CLIN_TEXTE]
    etapes = []
    if hrv:
        etapes.append(("hrv", Pipeline([
            ("imputation", SimpleImputer(strategy="median")),     # [PISTE] KNNImputer si beaucoup de trous
            ("correlation", FiltreCorrelation(SEUIL_CORR)),
            ("transfo", faire_transfo()),
        ]), hrv))
    if clin_num:
        etapes.append(("clin_num", Pipeline([
            ("imputation", SimpleImputer(strategy="most_frequent")),
            ("echelle", StandardScaler()),
        ]), clin_num))
    if clin_texte:
        etapes.append(("clin_texte", Pipeline([
            ("imputation", SimpleImputer(strategy="most_frequent")),
            ("codage", OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=False)),
            ("echelle", StandardScaler()),
        ]), clin_texte))
    return ColumnTransformer(etapes, verbose_feature_names_out=False)


SELECTIONS = {
    "toutes": lambda: "passthrough",
    # Importance de la forêt de référence (baisse moyenne d'impureté). [PISTE] TOP_K, ou une forêt plus grande.
    "sél. RF": lambda: SelectFromModel(
        RandomForestRegressor(n_estimators=200, max_features="sqrt", min_samples_leaf=5, random_state=0, n_jobs=1),
        max_features=lambda X: min(TOP_K, X.shape[1]), threshold=-np.inf),
    "PCA 90 %": lambda: PCA(n_components=VARIANCE_PCA, svd_solver="full"),
}

# 3. Models

Cinq familles de modèles. Le prof conseille surtout **LightGBM** et **XGBoost** (boosting d'arbres) ; Elastic Net reste la référence linéaire.

- Les hyperparamètres sensibles sont réglés par une **validation croisée interne** sur le pli d'entraînement (4 plis), en minimisant la MAE. Pour aller vite, le prétraitement et la sélection sont ajustés une fois sur tout le pli d'entraînement externe, puis la boucle interne ne règle que le modèle. Le pli de test externe n'est jamais vu, donc l'estimation finale reste honnête.
- Les arbres sont volontairement **peu profonds et très régularisés** : avec environ 145 enfants d'entraînement par pli, des arbres profonds apprennent le bruit.
- Chaque **expérience** = un modèle + une méthode de sélection + un jeu de variables. Toutes voient les mêmes enfants et les mêmes plis, donc leurs résultats sont directement comparables.

In [ ]:
INTERNE = KFold(4, shuffle=True, random_state=0)   # boucle interne : réglage des hyperparamètres

MODELES = {
    # selection="random" : environ 6 fois plus rapide ici (plus de variables que d'enfants), même choix d'hyperparamètres.
    "Elastic Net": lambda: ElasticNetCV(l1_ratio=L1_RATIOS, alphas=N_ALPHAS, eps=EPS_ALPHA, cv=INTERNE,
                                        max_iter=10_000, selection="random", random_state=0),
    # [PISTE] min_samples_leaf (3, 5, 10) et max_features ("sqrt", 0.3)
    "Random Forest": lambda: RandomForestRegressor(n_estimators=300, max_features="sqrt", min_samples_leaf=5,
                                                   random_state=0, n_jobs=1),
    # [PISTE] élargir les grilles si le réglage choisi est souvent au bord (section 6)
    "LightGBM": lambda: GridSearchCV(
        LGBMRegressor(learning_rate=0.03, min_child_samples=10, subsample=0.8, subsample_freq=1,
                      colsample_bytree=0.5, reg_lambda=1.0, random_state=0, n_jobs=1, verbose=-1),
        {"num_leaves": [4, 8], "n_estimators": [100, 300]}, cv=INTERNE, scoring="neg_mean_absolute_error"),
    "XGBoost": lambda: GridSearchCV(
        XGBRegressor(learning_rate=0.03, subsample=0.8, colsample_bytree=0.5, min_child_weight=5,
                     reg_lambda=1.0, random_state=0, n_jobs=1),
        {"max_depth": [2, 3], "n_estimators": [100, 300]}, cv=INTERNE, scoring="neg_mean_absolute_error"),
    # C et epsilon sont en points de QIT (la cible n'est pas standardisée).
    "SVR": lambda: GridSearchCV(SVR(kernel="rbf", gamma="scale"), {"C": [3, 10, 30], "epsilon": [1.0, 5.0]},
                                cv=INTERNE, scoring="neg_mean_absolute_error"),
}


def faire_pipeline(modele, selection, colonnes):
    return Pipeline([("prep", faire_preprocesseur(colonnes)), ("selection", SELECTIONS[selection]()),
                     ("modele", MODELES[modele]())])


JEUX = {"Clinique seul": cols_clin, "HRV seule": cols_lin + cols_nl, "Clinique + HRV": cols_clin + cols_lin + cols_nl}
# [PISTE] Une famille ou une période seule (suffixes : TD = temporel, FT = fréquentiel, GI = géométrique,
# 24h / Day / Night), par exemple :
# JEUX["HRV temporels"] = [c for c in cols_lin if "(TD" in c]
# JEUX["HRV nuit"] = [c for c in cols_lin + cols_nl if "night" in c.lower()]
# [PISTE] Idée du prof : des variables calculées sur les données brutes (fichiers RR). Les mettre dans un CSV
# (une ligne par enfant), le passer dans preparer_hrv(lire_csv(...), "brut"), joindre le résultat à `donnees`
# (section 1.5) et ajouter un jeu « Clinique + HRV + brut » ici.

# Expériences comparées : (modèle, sélection, jeu de variables).
EXPERIENCES = [
    ("Elastic Net", "toutes", "Clinique seul"),     # référence clinique : ce que la HRV doit améliorer
    ("Elastic Net", "toutes", "Clinique + HRV"),
    ("Random Forest", "toutes", "Clinique + HRV"),
    ("LightGBM", "sél. RF", "Clinique + HRV"),
    ("XGBoost", "sél. RF", "Clinique + HRV"),
    ("SVR", "PCA 90 %", "Clinique + HRV"),
    # [PISTE] ("Elastic Net", "PCA 90 %", "Clinique + HRV"), ("LightGBM", "toutes", "Clinique + HRV"),
    #         ("Elastic Net", "toutes", "HRV seule"), ("SVR", "sél. RF", "Clinique + HRV")...
]
EXPERIENCES = [e for e in EXPERIENCES if JEUX[e[2]]]

# 4. Evaluation

Chaque expérience est évaluée sur les plis communs (`folds.csv`) : pour chaque répétition et chaque pli, le modèle apprend sur les autres plis puis prédit le pli caché. Les prédictions sont écrites au format de `evaluate_model.py` dans `data/cleaned/predictions/`, ce qui permet de les comparer à celles de l'autre groupe.

- **4.1** lance les expériences ;
- **4.2** donne les T × K mesures demandées par le prof : MAE moyenne ± écart-type, MSE et R² ;
- **4.3** lance `evaluate()` de l'équipe : intervalles de confiance par bootstrap et comparaison appariée des modèles ;
- **4.4** et **4.5** regardent le meilleur modèle de plus près.

Avec `RAPIDE = True`, seules les 2 premières répétitions sont utilisées (10 mesures par modèle) : trop peu pour conclure, juste assez pour vérifier que tout tourne.

## 4.1 Lancement

In [ ]:
DOSSIER_PRED = DOSSIER_PARTAGE / "predictions"
DOSSIER_PRED.mkdir(parents=True, exist_ok=True)
REFERENCE = "QIT moyen (référence)"


def nom_experience(exp):
    return " · ".join(exp)


def details_modele(pipe):
    """Hyperparamètres choisis, variables sélectionnées et, pour Elastic Net, coefficients."""
    modele, selection = pipe.named_steps["modele"], pipe.named_steps["selection"]
    noms = pipe.named_steps["prep"].get_feature_names_out()
    details = {}
    if isinstance(selection, SelectFromModel):
        noms = noms[selection.get_support()]
        details["selection"] = list(noms)
    elif isinstance(selection, PCA):
        noms = np.array([f"composante {i + 1}" for i in range(selection.n_components_)])
        details["composantes"] = selection.n_components_
    if isinstance(modele, GridSearchCV):
        details["hyper"] = modele.best_params_
    elif isinstance(modele, ElasticNetCV):
        details["hyper"] = {"alpha": modele.alpha_, "l1_ratio": modele.l1_ratio_}
        details["coefs"] = pd.Series(modele.coef_, index=noms)
    return details


def ajuster_predire(exp, train_ids, test_ids, etiq):
    """Un pli : apprend sur les enfants train_ids, prédit test_ids (un modèle par cible)."""
    warnings.filterwarnings("ignore")
    modele, selection, jeu = exp
    colonnes = JEUX[jeu]
    predictions, details = {}, {}
    for cible in etiq.columns:
        pipe = faire_pipeline(modele, selection, colonnes).fit(X.loc[train_ids, colonnes], etiq.loc[train_ids, cible])
        predictions[cible] = np.ravel(pipe.predict(X.loc[test_ids, colonnes]))
        details[cible] = details_modele(pipe)
    return predictions, details


def taches_plis(repetitions=REPETITIONS):
    """(répétition, pli, enfants d'entraînement, enfants de test), découpés exactement comme dans run_cv."""
    taches = []
    for r in sorted(plis_partages["repeat"].unique()):
        if repetitions is not None and r not in repetitions:
            continue
        fr = plis_partages[plis_partages["repeat"] == r]
        for k in sorted(fr["fold"].unique()):
            taches.append((r, k, fr.loc[fr["fold"] != k, "child_id"].tolist(), fr.loc[fr["fold"] == k, "child_id"].tolist()))
    return taches


def lancer(exp, fichier):
    """Ajuste l'expérience sur tous les plis en parallèle, puis écrit les prédictions avec run_cv (format commun)."""
    taches = taches_plis()
    sorties = Parallel(n_jobs=N_JOBS)(delayed(ajuster_predire)(exp, tr, te, etiquettes) for _, _, tr, te in taches)
    deja_calcule = {tuple(te): pred for (_, _, _, te), (pred, _) in zip(taches, sorties)}
    run_cv(lambda tr, te: deja_calcule[tuple(te)], fichier_plis, fichier_cohorte, fichier,
           targets=CIBLES, repeats=REPETITIONS, verbose=False)
    return [details for _, details in sorties]


fichiers, details_exp = {}, {}
debut = time.time()
for exp in EXPERIENCES:
    t = time.time()
    nom = nom_experience(exp)
    fichiers[nom] = DOSSIER_PRED / f"predictions_{re.sub(r'[^a-z0-9]+', '_', normaliser_texte(nom)).strip('_')}.csv"
    details_exp[nom] = lancer(exp, fichiers[nom])
    print(f"{nom} ({len(JEUX[exp[2]])} variables) : {time.time() - t:.0f} s")
print(f"Total : {(time.time() - debut) / 60:.1f} min")

## 4.2 T × K mesures : MAE, MSE, R²

Une mesure par pli de test. Le R² d'un pli d'environ 36 enfants est très bruité ; le R² calculé sur toutes les prédictions hors pli (4.3) est plus fiable. La référence prédit, dans chaque pli, le QIT moyen des enfants d'entraînement : c'est le plancher à battre.

In [ ]:
def mesures_par_pli(vrai_pred):
    """vrai_pred : liste de (vrais QIT, QIT prédits), un élément par pli."""
    return pd.DataFrame([{"MAE": mean_absolute_error(v, p), "MSE": mean_squared_error(v, p), "R2": r2_score(v, p)}
                         for v, p in vrai_pred])


par_pli = {REFERENCE: mesures_par_pli([(etiquettes.loc[te, "QIT"], np.full(len(te), etiquettes.loc[tr, "QIT"].mean()))
                                       for _, _, tr, te in taches_plis()])}
for nom, fichier in fichiers.items():
    p = pd.read_csv(fichier)
    par_pli[nom] = mesures_par_pli([(g["true_QIT"], g["predicted_QIT"]) for _, g in p.groupby(["repeat", "fold"])])

ordre = sorted(par_pli, key=lambda n: par_pli[n]["MAE"].mean())
tableau = pd.DataFrame({n: {"mesures": len(m), "MAE": f"{m['MAE'].mean():.2f} ± {m['MAE'].std():.2f}",
                            "MSE": f"{m['MSE'].mean():.0f} ± {m['MSE'].std():.0f}",
                            "R²": f"{m['R2'].mean():.3f} ± {m['R2'].std():.3f}"} for n, m in par_pli.items()}).T
print(tableau.loc[ordre].to_string())

fig, ax = plt.subplots(figsize=(8, 0.45 * len(ordre) + 1.2))
for i, n in enumerate(ordre[::-1]):
    mae = par_pli[n]["MAE"]
    ax.scatter(mae, np.full(len(mae), i) + np.random.default_rng(i).uniform(-0.15, 0.15, len(mae)),
               s=10, alpha=0.35, color="tab:blue")
    ax.scatter(mae.mean(), i, s=60, color="black", zorder=3)
ax.axvline(par_pli[REFERENCE]["MAE"].mean(), ls="--", color="grey", label=REFERENCE)
ax.set_yticks(range(len(ordre)), ordre[::-1])
ax.set_xlabel("MAE par pli de test (points de QIT) ; point noir = moyenne")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

# Valeur ajoutée de la HRV par rapport au modèle clinique, pli par pli (mêmes plis, donc comparaison appariée).
seul, avec = "Elastic Net · toutes · Clinique seul", "Elastic Net · toutes · Clinique + HRV"
if seul in par_pli and avec in par_pli:
    gain = par_pli[seul]["MAE"] - par_pli[avec]["MAE"]
    print(f"\nAjouter la HRV au modèle clinique (Elastic Net) change la MAE de {-gain.mean():+.2f} points "
          f"(meilleur dans {(gain > 0).mean():.0%} des plis)")

## 4.3 Rapport commun de l'équipe (`evaluate_model.py`)

Pour chaque modèle : MAE et R² sur toutes les prédictions hors pli (moyenne ± écart-type entre répétitions), intervalles de confiance à 95 % par bootstrap sur les enfants, gain par rapport à la moyenne, puis comparaison appariée de chaque paire de modèles. **Si l'intervalle d'un gain contient 0, l'amélioration n'est pas démontrée.**

In [ ]:
resume_equipe, comparaison = evaluate(fichier_plis, fichier_cohorte, [str(f) for f in fichiers.values()],
                                      names=list(fichiers), targets=CIBLES)

## 4.4 Meilleur modèle : prédit contre observé

Un point = un enfant (sa prédiction quand il était dans le pli de test, 1re répétition). **Graphique individuel : ne pas le committer ni le partager.**

In [ ]:
MEILLEUR = next(n for n in ordre if n != REFERENCE)
exp_meilleur = next(e for e in EXPERIENCES if nom_experience(e) == MEILLEUR)
hors_pli = pd.read_csv(fichiers[MEILLEUR])
hors_pli = hors_pli[hors_pli["repeat"] == hors_pli["repeat"].min()]
vrai, pred = hors_pli["true_QIT"].to_numpy(), hors_pli["predicted_QIT"].to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
bornes = [min(vrai.min(), pred.min()) - 2, max(vrai.max(), pred.max()) + 2]
axes[0].scatter(vrai, pred, s=12, alpha=0.6)
axes[0].plot(bornes, bornes, "k--", lw=1, label="prédiction parfaite")
axes[0].axhline(vrai.mean(), color="grey", lw=1, label="QIT moyen")
axes[0].set(xlabel="QIT observé", ylabel="QIT prédit", title=MEILLEUR)
axes[0].legend()
axes[1].scatter(pred, vrai - pred, s=12, alpha=0.6)
axes[1].axhline(0, color="k", ls="--", lw=1)
axes[1].set(xlabel="QIT prédit", ylabel="résidu (observé − prédit)", title="Résidus")
plt.tight_layout()
plt.show()
# Si les points forment une bande horizontale autour du QIT moyen, le modèle prédit presque toujours la moyenne :
# il n'a pas trouvé de signal exploitable.

## 4.5 Test de permutation

On mélange les QIT entre les enfants et on refait toute la procédure (1re répétition). Si le vrai modèle ne fait pas nettement mieux que les modèles entraînés sur des QIT mélangés, sa performance ne se distingue pas du hasard.

In [ ]:
taches_1 = taches_plis([1])
melanges = [pd.DataFrame({"QIT": np.random.default_rng(GRAINE + i).permutation(etiquettes["QIT"].to_numpy())},
                         index=etiquettes.index) for i in range(N_PERMUTATIONS)]
sorties = Parallel(n_jobs=N_JOBS)(delayed(ajuster_predire)(exp_meilleur, tr, te, em)
                                  for em in melanges for _, _, tr, te in taches_1)
mae_melange = []
for i, em in enumerate(melanges):
    pred_m = pd.concat([pd.Series(sorties[i * len(taches_1) + j][0]["QIT"], index=te) for j, (_, _, _, te) in enumerate(taches_1)])
    mae_melange.append(mean_absolute_error(em.loc[pred_m.index, "QIT"], pred_m))
mae_melange = np.array(mae_melange)
mae_vraie = mean_absolute_error(vrai, pred)
p_valeur = (1 + np.sum(mae_melange <= mae_vraie)) / (1 + N_PERMUTATIONS)
print(f"{MEILLEUR} : MAE {mae_vraie:.2f} contre {mae_melange.mean():.2f} ± {mae_melange.std():.2f} avec QIT mélangés, "
      f"p = {p_valeur:.3f}")
# Avec RAPIDE = True (20 permutations), p ne peut pas descendre sous 0.048 : relancer avec RAPIDE = False.

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(mae_melange, bins=20, color="lightgrey", edgecolor="grey")
ax.axvline(mae_vraie, color="tab:red", lw=2, label=f"vrai modèle (p = {p_valeur:.3f})")
ax.set(xlabel="MAE (points de QIT)", ylabel="permutations")
ax.legend()
plt.tight_layout()
plt.show()

# 5. Prédiction conforme

Le modèle donne un QIT prédit, mais pas sa marge d'erreur. La **prédiction conforme** donne un intervalle [ŷ − q, ŷ + q] qui contient le vrai QIT avec une probabilité d'au moins 1 − α (ici 90 %), quel que soit le modèle :

1. sur les enfants d'entraînement, on mesure les erreurs du modèle par validation croisée (erreurs « hors pli », donc honnêtes) ;
2. q est le quantile 1 − α de ces erreurs, avec une petite correction pour l'effectif ;
3. pour un nouvel enfant, l'intervalle est sa prédiction ± q.

La garantie est vérifiée sur les plis de test de la 1re répétition : la part des enfants dont le vrai QIT tombe dans leur intervalle doit être proche de 90 %. La **largeur** dit si le modèle est utile : si elle est à peine plus petite que celle de l'intervalle construit autour du QIT moyen, le modèle n'apporte presque rien.

In [ ]:
def quantile_conforme(residus, alpha=ALPHA):
    n = len(residus)
    return np.quantile(residus, min(1.0, np.ceil((n + 1) * (1 - alpha)) / n), method="higher")


def intervalle_un_pli(exp, train_ids, test_ids):
    """Erreurs hors pli sur l'entraînement (validation croisée interne) -> demi-largeur q, puis prédiction du test."""
    etiq = etiquettes[["QIT"]]
    residus = []
    for tr_i, va_i in KFold(K_PLIS, shuffle=True, random_state=0).split(train_ids):
        tr_ids, va_ids = [train_ids[i] for i in tr_i], [train_ids[i] for i in va_i]
        pred_va, _ = ajuster_predire(exp, tr_ids, va_ids, etiq)
        residus.extend(np.abs(etiq.loc[va_ids, "QIT"].to_numpy() - pred_va["QIT"]))
    pred_te, _ = ajuster_predire(exp, train_ids, test_ids, etiq)
    y_train = etiq.loc[train_ids, "QIT"]
    return pd.DataFrame({"vrai": etiq.loc[test_ids, "QIT"].to_numpy(), "pred": pred_te["QIT"],
                         "q": quantile_conforme(np.array(residus)),
                         "moyenne": y_train.mean(), "q_ref": quantile_conforme(np.abs(y_train - y_train.mean()).to_numpy())},
                        index=test_ids)


conforme = pd.concat(Parallel(n_jobs=N_JOBS)(delayed(intervalle_un_pli)(exp_meilleur, tr, te) for _, _, tr, te in taches_1))
couverture = (np.abs(conforme["vrai"] - conforme["pred"]) <= conforme["q"]).mean()
couverture_ref = (np.abs(conforme["vrai"] - conforme["moyenne"]) <= conforme["q_ref"]).mean()
print(f"{MEILLEUR}, intervalle à {1 - ALPHA:.0%} :")
print(f"  couverture observée {couverture:.0%} (objectif ≥ {1 - ALPHA:.0%}), "
      f"demi-largeur ± {conforme['q'].mean():.1f} points de QIT")
print(f"  autour du QIT moyen : couverture {couverture_ref:.0%}, demi-largeur ± {conforme['q_ref'].mean():.1f} points")

# Pour un nouvel enfant : prédiction ± q, avec q calculé sur les erreurs hors pli de tous les enfants (1re répétition).
q_final = quantile_conforme(np.abs(vrai - pred))
print(f"\nPour un nouvel enfant : QIT prédit ± {q_final:.1f} points, avec une probabilité d'au moins {1 - ALPHA:.0%} "
      "de contenir son vrai QIT")

# Graphique individuel (un trait = un enfant) : ne pas le committer ni le partager.
tri = conforme.sort_values("pred")
dedans = np.abs(tri["vrai"] - tri["pred"]) <= tri["q"]
fig, ax = plt.subplots(figsize=(9, 4))
ax.vlines(range(len(tri)), tri["pred"] - tri["q"], tri["pred"] + tri["q"], color="lightsteelblue", lw=2)
ax.scatter(range(len(tri)), tri["vrai"], s=10, c=np.where(dedans, "black", "tab:red"), zorder=3)
ax.axhline(90, color="grey", ls=":", lw=1)
ax.set(xlabel="enfants triés par QIT prédit", ylabel="QIT", title=f"Intervalles à {1 - ALPHA:.0%} (rouge : vrai QIT en dehors)")
plt.tight_layout()
plt.show()

# 6. Explication

Deux questions :
- **Quelles variables le modèle utilise-t-il ?** Importance par permutation : on mélange une variable chez les enfants de test et on regarde de combien la MAE augmente. Elle est calculée sur des enfants que le modèle n'a pas vus (1re répétition), par variable d'origine, donc elle vaut pour tous les modèles, même après une PCA.
- **Ces choix sont-ils stables ?** Une variable importante dans un seul pli relève probablement du bruit ; une variable importante dans presque tous les plis est plus crédible.

In [ ]:
def importance_un_pli(exp, train_ids, test_ids):
    warnings.filterwarnings("ignore")
    modele, selection, jeu = exp
    colonnes = JEUX[jeu]
    pipe = faire_pipeline(modele, selection, colonnes).fit(X.loc[train_ids, colonnes], etiquettes.loc[train_ids, "QIT"])
    r = permutation_importance(pipe, X.loc[test_ids, colonnes], etiquettes.loc[test_ids, "QIT"],
                               scoring="neg_mean_absolute_error", n_repeats=5, random_state=0)
    return pd.Series(r.importances_mean, index=colonnes)


importances = pd.concat(Parallel(n_jobs=N_JOBS)(delayed(importance_un_pli)(exp_meilleur, tr, te)
                                                for _, _, tr, te in taches_1), axis=1)
resume_imp = pd.DataFrame({"hausse de MAE (points)": importances.mean(axis=1).round(3),
                           "% des plis > 0": (100 * (importances > 0).mean(axis=1)).round(0)})
resume_imp = resume_imp.sort_values("hausse de MAE (points)", ascending=False)
print(f"{MEILLEUR} : variables dont le mélange dégrade le plus la prédiction")
print(resume_imp.head(15).to_string())

haut = resume_imp.head(15)[::-1]
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh(haut.index, haut["hausse de MAE (points)"], color="tab:blue")
ax.axvline(0, color="k", lw=0.8)
ax.set(xlabel="hausse de la MAE quand la variable est mélangée (points de QIT)", title="Importance par permutation")
plt.tight_layout()
plt.show()

# Stabilité et réglages choisis dans les plis de l'évaluation (section 4).
details = [d["QIT"] for d in details_exp[MEILLEUR]]
if any("coefs" in d for d in details):
    coefs = pd.DataFrame([d["coefs"] for d in details if "coefs" in d])   # NaN = variable retirée par le filtre de corrélation
    frequence = (coefs.fillna(0) != 0).mean()
    print(f"\nElastic Net : {(frequence >= 0.8).sum()} variables retenues dans au moins 80 % des plis, "
          f"{(coefs.fillna(0) == 0).all(axis=1).sum()} plis sans aucune variable retenue")
    print(pd.DataFrame({"% des plis": (100 * frequence).round(0), "coef moyen": coefs.mean().round(2)})
          .sort_values("% des plis", ascending=False).head(15).to_string())
if any("selection" in d for d in details):
    frequence = pd.Series([v for d in details for v in d["selection"]]).value_counts() / len(details)
    print(f"\nSélection par la forêt : variables gardées le plus souvent (sur {len(details)} plis)")
    print((100 * frequence).round(0).head(15).to_string())
if any("composantes" in d for d in details):
    print(f"\nPCA : {min(d['composantes'] for d in details)} à {max(d['composantes'] for d in details)} composantes "
          f"pour garder {VARIANCE_PCA:.0%} de la variance")
hyper = pd.DataFrame([d["hyper"] for d in details if "hyper" in d])
if "alpha" in hyper:          # Elastic Net : alpha est continu, on le résume
    print(f"\nl1_ratio choisi : {hyper['l1_ratio'].value_counts().sort_index().to_dict()} ; alpha : médiane "
          f"{hyper['alpha'].median():.3g} (de {hyper['alpha'].min():.3g} à {hyper['alpha'].max():.3g})")
elif len(hyper):
    print(f"\nHyperparamètres choisis (sur {len(hyper)} plis) :",
          hyper.astype(str).apply(lambda r: ", ".join(f"{k}={v}" for k, v in r.items()), axis=1).value_counts().to_dict())
# [PISTE] Un réglage toujours au bord de sa grille (l1_ratio = 1.0, n_estimators = 300...) : élargir la grille de ce côté.
# Si Elastic Net ne retient presque aucune variable, le modèle ne trouve rien : c'est un résultat, pas un bug.

**Ce que ces résultats permettent de dire, et ce qu'ils ne permettent pas**

- Une variable importante est **associée** au QIT dans cette cohorte ; cela ne veut pas dire qu'elle le **cause**. La HRV néonatale est un marqueur de la maturation du système nerveux autonome, pas un levier sur lequel agir.
- La question « que faire pour un enfant prédit sous 90 ? » est une question **causale** : un modèle prédictif ne peut pas y répondre. Il peut au mieux **repérer** les enfants à suivre de plus près (dépistage, orientation vers un bilan), à condition que ses prédictions soient fiables, ce que les sections 4 et 5 permettent de juger. Les interventions relèvent d'études cliniques.
- Avec un signal faible, les importances changent beaucoup d'un pli à l'autre : ne pas surinterpréter une variable isolée.

# 7. Suite

- **Meilleurs réglages** : les `[PISTE]` de la configuration (`TOP_K`, `VARIANCE_PCA`, `SEUIL_CORR`, `PERIODES`) et des grilles de la section 3, toujours jugés sur la section 4.
- **Variables issues des données brutes** : voir le `[PISTE]` de la section 3.
- **Cibles secondaires** : `CIBLES = ["QIV", "QIP", "QIT"]`.
- **Explication plus fine des arbres** : valeurs SHAP (`pip install shap`, puis `shap.TreeExplainer` sur LightGBM ou XGBoost).
- **Classification** : QIT < 90 contre ≥ 90, avec AUC et kappa de Cohen (l'exactitude seule est trompeuse avec 78 % de la classe majoritaire).